In [1]:
!pip install -q gdown

import gdown
import os
import zipfile
import shutil
import random

file_id = "1PwZXse2huPI_weaKCSq21Olu2ButPFyM"
output = "/kaggle/working/data.zip"

gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

print("Download completed")



extract_path = "/kaggle/working/data"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(output, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed")


paths_to_delete = [
    "/kaggle/working/data/.config",
    "/kaggle/working/data/.ipynb_checkpoints",
    "/kaggle/working/data/all_folders.zip",
    "/kaggle/working/data/drive",
    "/kaggle/working/.virtual_documents"
]

for path in paths_to_delete:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    elif os.path.isfile(path):
        os.remove(path)
        print(f"Deleted file: {path}")
    else:
        print(f"Path not found (skipped): {path}")


def split_classes_train_val(
    data_dir,
    train_ratio=0.8,
    seed=42,
    extensions=None
):
    """
    Splits each class folder into train/ and validation/ subfolders.

    Structure BEFORE:
    data/
        class1/
            img1.jpg
            img2.jpg
        class2/
            img1.jpg
            img2.jpg

    Structure AFTER:
    data/
        class1/
            train/
            validation/
        class2/
            train/
            validation/
    """

    random.seed(seed)

    classes = [
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ]

    for cls in classes:
        class_path = os.path.join(data_dir, cls)

        files = [
            f for f in os.listdir(class_path)
            if os.path.isfile(os.path.join(class_path, f))
        ]

        if extensions:
            files = [f for f in files if f.lower().endswith(extensions)]

        if len(files) == 0:
            continue

        random.shuffle(files)

        split_idx = int(len(files) * train_ratio)
        train_files = files[:split_idx]
        val_files = files[split_idx:]

        train_dir = os.path.join(class_path, "train")
        val_dir = os.path.join(class_path, "validation")

        os.makedirs(train_dir, exist_ok=True)
        os.makedirs(val_dir, exist_ok=True)

        for f in train_files:
            shutil.move(
                os.path.join(class_path, f),
                os.path.join(train_dir, f)
            )

        for f in val_files:
            shutil.move(
                os.path.join(class_path, f),
                os.path.join(val_dir, f)
            )

        print(
            f"{cls}: "
            f"{len(train_files)} train, "
            f"{len(val_files)} validation"
        )
data_dir = "/kaggle/working/data"

IMAGE_EXTENSIONS = (
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp",
    ".ppm", ".pgm", ".pbm"
)

split_classes_train_val(
    data_dir=data_dir,
    train_ratio=0.8,
    extensions=IMAGE_EXTENSIONS
)



def keep_only_train_val(data_dir):
    allowed = {"train", "validation"}

    classes = [
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ]

    for cls in classes:
        class_path = os.path.join(data_dir, cls)

        for item in os.listdir(class_path):
            item_path = os.path.join(class_path, item)

            if os.path.isdir(item_path) and item not in allowed:
                shutil.rmtree(item_path)
                print(f"Deleted: {item_path}")

data_dir = "/kaggle/working/data"
keep_only_train_val(data_dir)

Downloading...
From (original): https://drive.google.com/uc?id=1PwZXse2huPI_weaKCSq21Olu2ButPFyM
From (redirected): https://drive.google.com/uc?id=1PwZXse2huPI_weaKCSq21Olu2ButPFyM&confirm=t&uuid=e3066c4b-1bb3-4cfe-8a55-76beaacf92eb
To: /kaggle/working/data.zip
100%|██████████| 993M/993M [00:10<00:00, 94.3MB/s] 


Download completed
Extraction completed
Deleted folder: /kaggle/working/data/.config
Deleted folder: /kaggle/working/data/.ipynb_checkpoints
Deleted file: /kaggle/working/data/all_folders.zip
Deleted folder: /kaggle/working/data/drive
Deleted folder: /kaggle/working/.virtual_documents
Vacuum_cleaner_dataset: 800 train, 200 validation
coffe maker_dataset: 800 train, 200 validation
Waffle iron_dataset: 800 train, 200 validation
cleaver_dataset: 800 train, 200 validation
Crock pot_dataset: 800 train, 200 validation
stove_dataset: 800 train, 200 validation
Hand Blower_dataset: 800 train, 200 validation
Salt Shaker_dataset: 800 train, 200 validation
chair_dataset: 800 train, 200 validation
microwave_dataset: 800 train, 200 validation
Chiffonier_dataset: 800 train, 200 validation
carpet_dataset: 800 train, 200 validation
Frying pan_dataset: 800 train, 200 validation
Dishwasher_dataset: 800 train, 200 validation
Pitcher_dataset: 800 train, 200 validation
Electric_fan_dataset: 800 train, 200 v

In [2]:
import tensorflow as tf

IMG_SIZE = 224  # ConvNeXt V2 input size

def load_dataset(data_dir, subset="train"):
    """
    Reads images from `data_dir/*/train` or `data_dir/*/validation`.
    Returns lists of image paths and corresponding labels.
    """
    images = []
    labels = []

    class_names = sorted(os.listdir(data_dir))
    class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}

    for cls_name in class_names:
        folder = os.path.join(data_dir, cls_name, subset)
        if not os.path.exists(folder):
            continue
        for f in os.listdir(folder):
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")):
                images.append(os.path.join(folder, f))
                labels.append(class_to_idx[cls_name])

    return images, labels, class_names


2025-12-21 20:02:30.735369: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766347350.934943      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766347350.991424      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766347351.458532      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766347351.458568      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766347351.458571      55 computation_placer.cc:177] computation placer alr

In [3]:
def create_dataset(image_paths, labels, batch_size=32, shuffle=True):
    path_ds = tf.data.Dataset.from_tensor_slices(image_paths)
    label_ds = tf.data.Dataset.from_tensor_slices(labels)

    def load_and_preprocess(path, label):
        # Read image
        image = tf.io.read_file(path)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
        image = tf.cast(image, tf.float32) / 255.0
        # Normalize for ConvNeXt V2
        mean = tf.constant([0.485, 0.456, 0.406])
        std  = tf.constant([0.229, 0.224, 0.225])
        image = (image - mean) / std
        return image, label

    ds = tf.data.Dataset.zip((path_ds, label_ds))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=len(image_paths))

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


In [4]:
train_images, train_labels, class_names = load_dataset(data_dir, subset="train")
val_images, val_labels, _ = load_dataset(data_dir, subset="validation")

train_ds = create_dataset(train_images, train_labels, batch_size=32, shuffle=True)
val_ds   = create_dataset(val_images, val_labels, batch_size=32, shuffle=False)

I0000 00:00:1766347365.076276      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [13]:
import tensorflow as tf

# number of classes
num_classes = len(class_names)

# =========================
# Create ConvNeXt Base model for transfer learning
# =========================
base_model = tf.keras.applications.ConvNeXtBase(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    pooling='avg'
)

# Freeze base model initially
base_model.trainable = False

# Add classification head
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)  # freeze BN layers
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

# =========================
# Compile model
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_convnext.keras",
        save_best_only=True,
        monitor="val_accuracy"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    )
]


# =========================
# Train (transfer learning)
# =========================
initial_epochs = 20
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=initial_epochs,
    callbacks=callbacks
)

print("Training complete.")


Epoch 1/20
624/624 ━━━━━━━━━━━━━━━━━━━━ 224s 282ms/step - accuracy: 0.5009 - loss: 2.1258 - val_accuracy: 0.8992 - val_loss: 0.5199
Epoch 2/20
624/624 ━━━━━━━━━━━━━━━━━━━━ 184s 253ms/step - accuracy: 0.9107 - loss: 0.4504 - val_accuracy: 0.9178 - val_loss: 0.3548
Epoch 3/20
624/624 ━━━━━━━━━━━━━━━━━━━━ 182s 253ms/step - accuracy: 0.9299 - loss: 0.3152 - val_accuracy: 0.9278 - val_loss: 0.2956
Epoch 4/20
624/624 ━━━━━━━━━━━━━━━━━━━━ 182s 253ms/step - accuracy: 0.9384 - loss: 0.2589 - val_accuracy: 0.9339 - val_loss: 0.2643
Epoch 5/20
624/624 ━━━━━━━━━━━━━━━━━━━━ 183s 254ms/step - accuracy: 0.9433 - loss: 0.2366 - val_accuracy: 0.9379 - val_loss: 0.2448
Epoch 6/20
624/624 ━━━━━━━━━━━━━━━━━━━━ 182s 252ms/step - accuracy: 0.9475 - loss: 0.2118 - val_accuracy: 0.9385 - val_loss: 0.2316
Epoch 7/20
624/624 ━━━━━━━━━━━━━━━━━━━━ 182s 252ms/step - accuracy: 0.9527 - loss: 0.1953 - val_accuracy: 0.9421 - val_loss: 0.2204
Epoch 8/20
624/624 ━━━━━━━━━━━━━━━━━━━━ 182s 253ms/step - accuracy: 0.9559 -

In [14]:
model = tf.keras.models.load_model("best_convnext.keras")
